# Convallaria PN2V with a Gaussian mixture noise model

This notebook follows the PPN2V Convallaria protocol
([arXiv:1911.12291](https://arxiv.org/abs/1911.12291)).
Both noise models are Gaussian mixtures fit with `NoiseModelTrainer`.
Calibration uses the mean of the static diaphragm stack as the signal.
Bootstrap trains N2V on the noisy movie and uses that prediction as the signal.
The reported image is the PN2V MMSE estimate.

Paper targets on the top-left 512×512 crop, GT = mean of the 100 noisy frames:

- calibration GMM: 36.47 dB
- bootstrap GMM: 36.70 dB

Data: [Zenodo 5156913](https://zenodo.org/record/5156913), `Convallaria_diaphragm.zip`.

In [ ]:
from pathlib import Path

import numpy as np
import tifffile
from lightning.pytorch import seed_everything

from careamics import CAREamist, NoiseModelTrainer
from careamics.config import create_advanced_n2v_config, create_advanced_pn2v_config
from careamics.config.noise_model import GaussianMixtureNMConfig

seed_everything(42)

DATA_DIR = Path("demos/Convallaria_diaphragm")
if not DATA_DIR.exists():
    DATA_DIR = Path("Convallaria_diaphragm")
NOISY_NAME = "20190520_tl_25um_50msec_05pc_488_130EM_Conv.tif"
CALIBRATION_NAME = "20190726_tl_50um_500msec_wf_130EM_FD.tif"

noisy = tifffile.imread(DATA_DIR / NOISY_NAME).astype(np.float32)
calibration = tifffile.imread(DATA_DIR / CALIBRATION_NAME).astype(np.float32)
noisy.shape, calibration.shape

In [ ]:
def paper_psnr(ground_truth: np.ndarray, prediction: np.ndarray) -> float:
    """PSNR with data range max(gt) - min(gt), as in the PPN2V notebooks."""
    data_range = float(ground_truth.max() - ground_truth.min())
    mse = float(np.mean((ground_truth - prediction) ** 2))
    return 20.0 * np.log10(data_range) - 10.0 * np.log10(mse)


def convallaria_score(prediction: np.ndarray) -> float:
    """Mean per-frame PSNR on the top-left 512 crop.

    Ground truth is the mean of the 100 noisy frames. ``prediction`` is SYX.
    """
    frames = prediction[:, :512, :512]
    ground_truth = noisy[:, :512, :512].mean(axis=0)
    return float(np.mean([paper_psnr(ground_truth, frame) for frame in frames]))


train_stack = noisy[:-5]
val_stack = noisy[-5:]
norm_mean = float(noisy.mean())
norm_std = float(noisy.std())
norm_mean, norm_std

## Shared PN2V settings

Depth, filter count, and the 800-sample head are `UNetConfig` fields.
The UNet implementation is unchanged.

In [ ]:
PN2V_MODEL = {"depth": 3, "num_channels_init": 64}
PN2V_TRAINER = {"accumulate_grad_batches": 20}
PN2V_SCHEDULER = {"factor": 0.5, "patience": 10}


def train_gmm(signal: np.ndarray, observation: np.ndarray) -> GaussianMixtureNMConfig:
    """Fit one GMM (K=3, n=2, min_sigma=50) and return its config."""
    trainer = NoiseModelTrainer(n_gaussian=3, n_coeff=2, min_sigma=50.0)
    trainer.train_from_pairs(
        signal=signal,
        observation=observation,
        axes="SYX",
        n_epochs=2000,
        learning_rate=1e-1,
        batch_size=250000,
    )
    return trainer.get_config().noise_models[0]


def train_pn2v(noise_model: GaussianMixtureNMConfig, name: str) -> CAREamist:
    # Depth 3 needs a spatial size divisible by 8. Upstream uses 100.
    config = create_advanced_pn2v_config(
        experiment_name=name,
        data_type="array",
        axes="SYX",
        patch_size=[96, 96],
        batch_size=1,
        noise_model=noise_model,
        num_epochs=200,
        num_steps=5,
        num_classes=800,
        n_channels=1,
        roi_size=5,
        masked_pixel_percentage=3.125,
        model_params=PN2V_MODEL,
        trainer_params=PN2V_TRAINER,
        optimizer_params={"lr": 1e-3},
        lr_scheduler_params=PN2V_SCHEDULER,
        normalization_params={
            "input_means": [norm_mean],
            "input_stds": [norm_std],
        },
        n_val_patches=1,
    )
    engine = CAREamist(config)
    engine.train(train_data=train_stack, val_data=val_stack)
    return engine

## Calibration GMM

Signal is the mean of the 100 static calibration frames. Observations are those frames.

In [ ]:
calibration_signal = np.repeat(
    calibration.mean(axis=0, keepdims=True), calibration.shape[0], axis=0
)
calibration_nm = train_gmm(calibration_signal, calibration)
calibration_engine = train_pn2v(calibration_nm, "convallaria_pn2v_gmm")
calibration_prediction, _ = calibration_engine.predict(
    noisy, tile_size=(192, 192), tile_overlap=(48, 48)
)
convallaria_score(calibration_prediction[0])

## Bootstrap GMM

Train N2V on the noisy movie, predict it, and pass that prediction to the same
`NoiseModelTrainer.train_from_pairs`. There is no separate bootstrap class.

The copied trainer uses the full signal range. The original bootstrap notebook
also clips the signal at percentiles 0.5 and 99.5.

In [ ]:
n2v_config = create_advanced_n2v_config(
    experiment_name="convallaria_n2v",
    data_type="array",
    axes="SYX",
    patch_size=[96, 96],
    batch_size=1,
    num_epochs=200,
    num_steps=10,
    n_channels=1,
    roi_size=5,
    masked_pixel_percentage=3.125,
    model_params=PN2V_MODEL,
    trainer_params=PN2V_TRAINER,
    optimizer_params={"lr": 1e-3},
    lr_scheduler_params=PN2V_SCHEDULER,
    normalization_params={
        "input_means": [norm_mean],
        "input_stds": [norm_std],
    },
    n_val_patches=1,
)
n2v_engine = CAREamist(n2v_config)
n2v_engine.train(train_data=train_stack, val_data=val_stack)
n2v_prediction, _ = n2v_engine.predict(
    noisy, tile_size=(192, 192), tile_overlap=(48, 48)
)
n2v_prediction = n2v_prediction[0].astype(np.float32)
if n2v_prediction.ndim == 4:
    n2v_prediction = n2v_prediction[:, 0]

bootstrap_nm = train_gmm(n2v_prediction, noisy)
bootstrap_engine = train_pn2v(bootstrap_nm, "convallaria_pn2v_boot_gmm")
bootstrap_prediction, _ = bootstrap_engine.predict(
    noisy, tile_size=(192, 192), tile_overlap=(48, 48)
)
convallaria_score(bootstrap_prediction[0])